# Ranking the effectiveness of senolytic drugs on gliomas
**Reference:** **Tibor Fekete, J., & Győrffy, B. (2022). A unified platform enabling biomarker ranking and validation for 1562 drugs using transcriptomic data of 1250 cancer cell lines.** _Computational and Structural Biotechnology Journal_, 20, **2885–2894**. https://doi.org/10.1016/j.csbj.2022.06.007
 @tiborfeketeUnifiedPlatformEnabling2022

This notebook implements the workflow described by Fekete and Győrffy (2022):
1. Filter expression data for gliomas.
2. Label the lower response tertile as **sensitive** and the upper tertile as **resistant**; omit the middle tertile. _(Or something smarter)_
3. Randomly split samples into 66% training and 34% testing sets.
4. Select candidate genes with a training-set Mann-Whitney U test at $p<0.05$.
5. Combine selected genes in a random-forest classifier.
6. Evaluate the untouched test set using ROC AUC, accuracy, Cohen's kappa, sensitivity, specificity, and precision.

The implementation also includes single-gene ROC/Mann-Whitney results, Spearman correlations, Benjamini-Hochberg FDR, cell-line ranking, and a correlation matrix.

In [1]:
import pandas as pd
import numpy as np
import qnorm
from scipy.stats import mannwhitneyu
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, precision_score

## 1. Load and filter expression data

- Model.cvs was used for identification of cell line oncotree lineages
- Source: https://depmap.org/portal/data_page/?tab=allData&releasename=DepMap%20Public%2026Q1&filename=Model.csv
- Two expression datasets were used:

### 1.1 CCLE/DepMap/CTRP RNA-seq
- **Raw Data:** `Expression_(Short-read)_Public_26Q1_subsetted.csv`
- **Source:** https://depmap.org/portal/download/all/?release=DepMap+Public+26Q1&file=OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv&tab=allData

In [2]:
# Data import
ccle_exp = pd.read_csv(
    "../raw_data/expression/glioma/Expression_(Short-read)_Public_26Q1_subsetted.csv",
    index_col=0
)

In [3]:
ccle_exp.iloc[:5, :5]

,RBM5,RBM6,SLC25A13,CACNG3,ZNF263
ACH-001113,5.591937,5.578833,3.518019,0.000000,4.832774
ACH-001289,6.323073,5.533597,4.864454,0.024853,5.095810
ACH-001339,5.248755,4.470589,5.362344,0.000000,3.891711
ACH-001979,5.015364,3.445246,4.436162,0.000000,5.127530
ACH-002438,5.085331,4.914588,3.542815,0.000000,4.618145


In [4]:
model = pd.read_csv(
    "../raw_data/annotation/Model.csv",
    index_col=0
)

In [5]:
model.iloc[:200, :200]

,PatientID,CellLineName,StrippedCellLineName,DepmapModelType,OncotreeLineage,OncotreePrimaryDisease,OncotreeSubtype,OncotreeCode,PatientSubtypeFeatures,RRID,...,PublicComments,CCLEName,HCMIID,PediatricModelType,ModelAvailableInDbgap,ModelSubtypeFeatures,WTSIMasterCellID,SangerModelID,COSMICID,ModelIDAlias
ModelID,,,,,,,,,,,,,,,,,,,,,
ACH-000001,PT-gj46wT,NIH:OVCAR-3,NIHOVCAR3,HGSOC,Ovary/Fallopian Tube,Ovarian Epithelial Tumor,High-Grade Serous Ovarian Cancer,HGSOC,NaN,CVCL_0465,...,NaN,NIHOVCAR3_OVARY,NaN,False,Approved for public sharing - CCLE,NaN,2201.0,SIDM00105,905933.0,NaN
ACH-000002,PT-5qa3uk,HL-60,HL60,AMLMRC,Myeloid,Acute Myeloid Leukemia,AML with Myelodysplasia-Related Changes,AMLMRC,"TP53(del), CDKN2A and NRAS mutations [PubMed=2...",CVCL_0002,...,NaN,HL60_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,True,Approved for public sharing - CCLE,"NRAS, BCOR and CDKN2A",55.0,SIDM00829,905938.0,NaN
ACH-000003,PT-puKIyc,CACO2,CACO2,COAD,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma,COAD,NaN,CVCL_0025,...,NaN,CACO2_LARGE_INTESTINE,NaN,False,Approved for public sharing - CCLE,NaN,NaN,SIDM00891,NaN,NaN
ACH-000004,PT-q4K2cp,HEL,HEL,AMLNOS,Myeloid,Acute Myeloid Leukemia,"AML, NOS",AMLNOS,JAK2 and TP53 mutations,CVCL_0001,...,NaN,HEL_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,True,Approved for public sharing - CCLE,JAK2 and TP53,783.0,SIDM00594,907053.0,NaN
ACH-000005,PT-q4K2cp,HEL 92.1.7,HEL9217,AML,Myeloid,Acute Myeloid Leukemia,Acute Myeloid Leukemia,AML,JAK2 and TP53 mutations,CVCL_2481,...,NaN,HEL9217_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,True,Approved for public sharing - CCLE,NaN,NaN,SIDM00593,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ACH-000197,PT-XChmHC,TALL-1,TALL1,TALL,Lymphoid,T-Lymphoblastic Leukemia/Lymphoma,T-Cell Acute Lymphoblastic Leukemia,NaN,"STAG2-PTGER3,UHRF1-PLIN3, NRAS and TP53 mutations",CVCL_1736,...,NaN,TALL1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,True,Approved for public sharing - CCLE,"TAL1, TP53",2218.0,SIDM00370,909762.0,NaN
ACH-000198,PT-FYT6ze,EOL-1,EOL1,AML,Myeloid,Acute Myeloid Leukemia,Acute Myeloid Leukemia,AML,"FIP1L1::PDGFRA, KMT2A partial tandem duplication",CVCL_0258,...,NaN,EOL1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,True,Approved for public sharing - CCLE,"FIP1L1::PDGFRA, KMT2A-PTD, PHF6",426.0,SIDM00277,906856.0,NaN
ACH-000199,PT-B6GmiY,Hs 255.T,HS255T,ZFIBCOL,Fibroblast,Non-Cancerous,"Fibroblast, Colorectal",NaN,NaN,CVCL_0707,...,NaN,HS255T_FIBROBLAST,NaN,False,Approved for public sharing - CCLE,NaN,NaN,SIDM01613,NaN,NaN


In [6]:
# Add metadata to the expression data
ccle_exp.index.name = "ModelID"

metadata_columns = [
    "OncotreeLineage",
    "OncotreePrimaryDisease",
    "OncotreeSubtype",
]

for position, column in enumerate(metadata_columns):
    ccle_exp.insert(
        position,
        column,
        model[column].reindex(ccle_exp.index),
    )

In [7]:
print("Unmatched models:", ccle_exp["OncotreeLineage"].isna().sum())

Unmatched models: 12


In [8]:
ccle_exp = ccle_exp.dropna(subset=["OncotreeLineage"])

In [9]:
ccle_exp.iloc[:200, :200]

,OncotreeLineage,OncotreePrimaryDisease,OncotreeSubtype,RBM5,RBM6,SLC25A13,CACNG3,ZNF263,CCL26,USH1C,...,DNPH1,PRSS16,KIF20A,CDH9,APBB3,CERT1,POLR3G,CDH6,SLC12A7,RAD50
ModelID,,,,,,,,,,,,,,,,,,,,,
ACH-001113,Lung,Non-Small Cell Lung Cancer,Lung Squamous Cell Carcinoma,5.591937,5.578833,3.518019,0.000000,4.832774,1.269291,0.276878,...,4.749384,3.794333,4.750681,0.000000,4.881063,4.414744,2.455860,0.925937,3.583952,4.232646
ACH-001289,CNS/Brain,Embryonal Tumor,Atypical Teratoid/Rhabdoid Tumor,6.323073,5.533597,4.864454,0.024853,5.095810,0.000000,0.073646,...,6.211497,3.481596,5.501506,0.034178,5.011482,5.107679,1.999762,0.038210,2.986154,5.269370
ACH-001339,Skin,Melanoma,Cutaneous Melanoma,5.248755,4.470589,5.362344,0.000000,3.891711,0.252796,0.012451,...,4.407971,0.303334,5.859637,0.019885,2.621977,5.772039,5.113521,0.080579,3.219269,3.643868
ACH-001979,Skin,Melanoma,Melanoma,5.015364,3.445246,4.436162,0.000000,5.127530,0.000000,0.000000,...,4.544739,2.710814,2.300305,0.000000,1.666729,5.370063,4.448095,0.000000,4.154761,4.860670
ACH-002438,Kidney,Wilms' Tumor,Wilms' Tumor,5.085331,4.914588,3.542815,0.000000,4.618145,1.969127,1.329652,...,4.330501,0.000000,5.917816,0.000000,2.236309,3.914182,1.939087,0.184972,2.799699,4.148999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ACH-000858,Lung,Non-Small Cell Lung Cancer,Lung Squamous Cell Carcinoma,4.625791,3.986620,4.781417,0.000000,4.269961,0.750315,0.000000,...,5.223295,2.493376,3.940308,0.000000,2.138097,3.462537,2.920495,6.089793,2.989889,3.362209
ACH-001421,Eye,Retinoblastoma,Retinoblastoma,6.484462,6.255697,5.637315,2.975227,5.194717,1.601354,0.000000,...,6.744941,0.301144,5.463171,0.000000,4.747216,3.749019,2.691695,0.000000,4.616210,4.180554
ACH-002796,Soft Tissue,CIC-DUX4 Sarcoma,CIC-DUX4 Sarcoma,6.103065,5.809877,4.242321,0.000000,4.816528,0.078420,0.000000,...,4.877906,0.618601,4.460205,0.631748,3.705867,4.902754,1.736730,1.875202,3.224340,1.240388


In [10]:
ccle_brain_exp = ccle_exp[
    ccle_exp["OncotreeLineage"].eq("CNS/Brain")
].copy()

In [11]:
print(ccle_brain_exp.shape)
print(ccle_brain_exp["OncotreeLineage"].value_counts())

(103, 19218)
OncotreeLineage
CNS/Brain    103
Name: count, dtype: int64


### 1.2 **GDSC1/GDSC2 RNA-normalized Affymetrix HGU-219 Expression Matrix:**
- As of 19/7/2026, the preprocessed expression data released by the Sanger Institute is not available on the original website anymore.
- This source was used to trace back the original link: https://github.com/BorgwardtLab/Kernelized-Rank-Learning/blob/master/URLs_to_GDSC_datasets.txt
- **Raw Data ArrayExpress Accession:** `E-MTAB-3610`
- **Preprocessed Data:** `Cell_line_RMA_proc_basalExp.txt`
- **Original Source:** http://www.cancerrxgene.org/gdsc1000/GDSC1000_WebResources/Data/preprocessed/Cell_line_RMA_proc_basalExp.txt.zip
- **Currently Available Source:** https://web.archive.org/web/20260209143748/http://www.cancerrxgene.org/gdsc1000/GDSC1000_WebResources/Data/preprocessed/Cell_line_RMA_proc_basalExp.txt.zip

In [12]:
# Data import
gdsc_exp = pd.read_csv(
    "../raw_data/expression/Cell_line_RMA_proc_basalExp.txt",
    sep="\t",
    low_memory=False
)

In [13]:
gdsc_exp.iloc[:5, :5]

,GENE_SYMBOLS,GENE_title,DATA.906826,DATA.687983,DATA.910927
0,TSPAN6,tetraspanin 6 [Source:HGNC Symbol;Acc:11858],7.632023,7.548671,8.712338
1,TNMD,tenomodulin [Source:HGNC Symbol;Acc:17757],2.964585,2.777716,2.643508
2,DPM1,dolichyl-phosphate mannosyltransferase polypep...,10.379553,11.807341,9.880733
3,SCYL3,SCY1-like 3 (S. cerevisiae) [Source:HGNC Symbo...,3.614794,4.066887,3.956230
4,C1orf112,chromosome 1 open reading frame 112 [Source:HG...,3.380681,3.732485,3.236620


In [14]:
gdsc_exp = gdsc_exp.dropna(subset=["GENE_SYMBOLS"]).copy()
gdsc_exp = gdsc_exp.set_index("GENE_SYMBOLS")
gdsc_exp = gdsc_exp.drop(columns=["GENE_title"])
gdsc_exp = gdsc_exp.T

gdsc_exp.index = (
    gdsc_exp.index.astype(str)
    .str.replace(r"^DATA\.", "", regex=True)
)

gdsc_exp.index.name = "COSMICID"

In [15]:
metadata_columns = [
    "OncotreeLineage",
    "OncotreePrimaryDisease",
    "OncotreeSubtype",
]

gdsc_exp.index = gdsc_exp.index.astype(str)

model_cosmic = model.dropna(subset=["COSMICID"]).copy()
model_cosmic["COSMICID"] = (
    model_cosmic["COSMICID"]
    .astype("Int64")
    .astype(str)
)

metadata = (
    model_cosmic
    .drop_duplicates(subset="COSMICID")
    .set_index("COSMICID")[metadata_columns]
    .reindex(gdsc_exp.index)
)

gdsc_exp = gdsc_exp.drop(
    columns=[col for col in metadata_columns if col in gdsc_exp.columns]
)

gdsc_exp = metadata.join(gdsc_exp)

gdsc_exp.index.name = "COSMICID"

In [16]:
gdsc_exp.iloc[:200, :5]

,OncotreeLineage,OncotreePrimaryDisease,OncotreeSubtype,TSPAN6,TNMD
COSMICID,,,,,
906826,Breast,Invasive Breast Carcinoma,Invasive Breast Carcinoma,7.632023,2.964585
687983,Lung,SMARCA4-deficient undifferentiated tumor,SMARCA4-deficient undifferentiated tumor,7.548671,2.777716
910927,Breast,Invasive Breast Carcinoma,Invasive Breast Carcinoma,8.712338,2.643508
1240138,Pleura,Pleural Mesothelioma,"Pleural Mesothelioma, Biphasic Type",7.797142,2.817923
1240139,Pleura,Pleural Mesothelioma,Pleural Mesothelioma,7.729268,2.957739
...,...,...,...,...,...
907300,Myeloid,Acute Myeloid Leukemia,Acute Myelomonocytic Leukemia,3.334582,2.709345
907276,Esophagus/Stomach,Esophagogastric Adenocarcinoma,Signet Ring Cell Carcinoma of the Stomach,7.448062,2.887007
907277,Lymphoid,T-Lymphoblastic Leukemia/Lymphoma,T-Cell Acute Lymphoblastic Leukemia,6.160155,2.907307


In [17]:
print("Unmatched models:", gdsc_exp["OncotreeLineage"].isna().sum())

Unmatched models: 73


In [18]:
gdsc_exp = gdsc_exp.dropna(subset=["OncotreeLineage"])

In [19]:
gdsc_brain_exp = gdsc_exp[
    gdsc_exp["OncotreeLineage"].eq("CNS/Brain")
].copy()

In [20]:
print(gdsc_brain_exp.shape)
print(gdsc_brain_exp["OncotreeLineage"].value_counts())

(56, 17422)
OncotreeLineage
CNS/Brain    56
Name: count, dtype: int64


## 1.3 Filtering
Reference: **Zhang, C., Neha, Zhang, J., Dhaha, P., Li, X., Mishra, S. K., Fleishman, J., Parvez, S., Jha, S. K., & Huang, M. (2025). Aging and senescence: Key players in brain tumor progression and drug resistance. Drug Resistance Updates, 81, 101228.** https://doi.org/10.1016/j.drup.2025.101228
 zhangAgingSenescenceKey2025

Primary types of gliomas according to Zhang et al.:
1. Astrocytomas: e.g.: Glioblastoma
2. Oligodendrogliomas
3. Ependymomas
4. Mixed gliomas


In [21]:
print("CCLE subtype counts:")
print(ccle_brain_exp["OncotreeSubtype"].value_counts(dropna=False))

print("\nGDSC subtype counts:")
print(gdsc_brain_exp["OncotreeSubtype"].value_counts(dropna=False))

CCLE subtype counts:
OncotreeSubtype
Glioblastoma, IDH-Wildtype                     55
Astrocytoma, IDH-Mutant                        13
Medulloblastoma, NOS                            7
Atypical Teratoid/Rhabdoid Tumor                6
Meningioma                                      4
Primitive Neuroectodermal Tumor                 3
Medulloblastoma, Group 3                        3
Immortalized Astrocytes                         3
Gliosarcoma                                     2
Adult-Type Diffuse Glioma                       2
Immortalized Epithelial Cells, CNS              1
Immortalized Endothelial Cells, Endoneurial     1
Immortalized Neuronal Progenitor Cells          1
Astrocytoma, IDH-Mutant, Grade 3                1
Glioma, NOS                                     1
Name: count, dtype: int64

GDSC subtype counts:
OncotreeSubtype
Glioblastoma, IDH-Wildtype          34
Astrocytoma, IDH-Mutant             11
Gliosarcoma                          4
Medulloblastoma, NOS            